Initialise ifcopenshell libraries and conditions

In [56]:
import numpy as np
import ifcopenshell

# for some reason ifcopenshell.api is not imported when importing ifcopenshell, so we need to import it explicitly
import ifcopenshell.api.root
import ifcopenshell.api.unit
import ifcopenshell.api.context
import ifcopenshell.api.project
import ifcopenshell.api.geometry

model = ifcopenshell.file()

# create the root entity (IfcProject) and set it as the root of the model
ifcopenshell.api.root.create_entity(model, ifc_class="IfcProject")

# set coordinates to be in meters
length = ifcopenshell.api.unit.add_si_unit(model, unit_type="LENGTHUNIT")
ifcopenshell.api.unit.assign_unit(model, units=[length])

# create contexts for 3D model, plan view and body
model3d = ifcopenshell.api.context.add_context(model, context_type="Model")
plan = ifcopenshell.api.context.add_context(model, context_type="Plan")
body = ifcopenshell.api.context.add_context(model, 
    context_type="Model", context_identifier="Body", target_view="MODEL_VIEW", parent=model3d)

# add some contexts for axes and annotations
ifcopenshell.api.context.add_context(model,
    context_type="Model", context_identifier="Axis", target_view="GRAPH_VIEW", parent=model3d)
ifcopenshell.api.context.add_context(model,
    context_type="Plan", context_identifier="Axis", target_view="GRAPH_VIEW", parent=plan)
ifcopenshell.api.context.add_context(model,
    context_type="Model", context_identifier="Box", target_view="MODEL_VIEW", parent=model3d)
ifcopenshell.api.context.add_context(model,
    context_type="Plan", context_identifier="Annotation", target_view="PLAN_VIEW", parent=plan)


# create the project structure components for use in IfcRelAggregates
project = model.by_type("IfcProject")[0]
# site = ifcopenshell.api.root.create_entity(model, ifc_class="IfcSite", name="Main Site")
# building = ifcopenshell.api.root.create_entity(model, ifc_class="IfcBuilding", name="Building")
site = ifcopenshell.api.root.create_entity(model, ifc_class="IfcSite", name="Main Site")
building = ifcopenshell.api.root.create_entity(model, ifc_class="IfcBuilding", name="Building")
storey = model.create_entity("IfcBuildingStorey", Name="Ground", Elevation=0.0)

# IfcRelAggregates is used to define the hierarchical structure of the building model.
model.create_entity("IfcRelAggregates", RelatingObject=project, RelatedObjects=[site])
model.create_entity("IfcRelAggregates", RelatingObject=site, RelatedObjects=[building])
model.create_entity("IfcRelAggregates", RelatingObject=building, RelatedObjects=[storey])


#23=IfcRelAggregates($,$,$,$,#19,(#20))

Create function to create IfcSpaces, Walls, Floors

In [ ]:
import profile

def add_space(length, width, name, origin, height = 3.0):
    
    # create the specific room ifcspace and relate it to the storey in the model hierarchy
    ifcspace = ifcopenshell.api.root.create_entity(model, ifc_class="IfcSpace", name=name)
    model.create_entity("IfcRelAggregates", RelatingObject=storey, RelatedObjects=[ifcspace])

    # create the actual space entity with geometry and relate it to the ifcspace in the model hierarchy
    room_space = ifcopenshell.api.root.create_entity(model, ifc_class="IfcSpace", name=name)
    # model.create_entity("IfcRelAggregates", RelatingObject=storey, RelatedObjects=[space])

    matrix = np.eye(4)
    matrix[:,3][0:3] = origin
    ifcopenshell.api.geometry.edit_object_placement(model, product=room_space, matrix=matrix, is_si=False)

    profile = model.create_entity("IfcRectangleProfileDef",
        ProfileName=f"{name} footprint", ProfileType="AREA", 
        XDim=length, YDim=width)
    
    space_rep = ifcopenshell.api.geometry.add_profile_representation(
        model, context=body, profile=profile, depth=height)
    
    #### this assigns the geometric representation to the ifcspace, which is what allows it to be visualized in a viewer.
    #### not actually needed so can hide. Maybe useful to keep it for debugging purposes though.
    # ifcopenshell.api.geometry.assign_representation(
    #     model, product=room_space, representation=space_rep)
    
    #### the below is supposed to assign a transparent style to the space. But doesn't work. Comes up as blue in the viewer
    # style = ifcopenshell.api.style.add_style(model)
    # ifcopenshell.api.style.add_surface_style(
    #     model,
    #     style=style,
    #     ifc_class="IfcSurfaceStyleShading",
    #     attributes={
    #         "SurfaceColour": {"Name": None, "Red": 0.7, "Green": 0.9, "Blue": 1.0},
    #         "Transparency": 1,  # 0 = opaque, 1 = fully transparent
    #     }
    # )
    # ifcopenshell.api.style.assign_representation_styles(model,
    #                 shape_representation=room_space.Representation.Representations[0],
    #                 styles=[style]
    #             )
    model.create_entity("IfcRelContainedInSpatialStructure", RelatingStructure=ifcspace, RelatedElements=[room_space])

    return ifcspace, room_space


# define a function to add ifcwall to the model
def add_wall(length, name: str, origin, rotate, ifcspace):
    wall = ifcopenshell.api.root.create_entity(model, ifc_class="IfcWall", name=name)
    
    wall_rep = ifcopenshell.api.geometry.add_wall_representation(model,
        context=body, length=length, height=3.0, thickness=0.2)
    
    matrix = np.eye(4)
    if rotate:
        matrix = ifcopenshell.util.placement.rotation(90, "Z") @ matrix
    matrix[:,3][0:3] = origin
    
    ifcopenshell.api.geometry.edit_object_placement(model, product=wall, matrix=matrix, is_si=False)
    ifcopenshell.api.geometry.assign_representation(model, product=wall, representation=wall_rep)

    # this part is important because it relates the wall back to the hierarchy of the ifc model defined before
    ifcopenshell.api.spatial.assign_container(model, relating_structure=ifcspace, products=[wall])

    return wall

def add_slab(length, width, name, origin, ifcspace):
    slab = ifcopenshell.api.root.create_entity(model, ifc_class="IfcSlab", name=name)

    # Create a rectangular profile for the slab with your desired dimensions
    profile = model.create_entity("IfcRectangleProfileDef", ProfileName="slab", ProfileType="AREA",
        XDim=length, YDim=width)

    slab_rep = ifcopenshell.api.geometry.add_profile_representation(model, context=body, profile=profile, depth=0.2)

    matrix = np.eye(4)
    matrix[:,3][0:3] = origin
    ifcopenshell.api.geometry.edit_object_placement(model, product=slab, matrix=matrix, is_si=True)
    ifcopenshell.api.geometry.assign_representation(model, product=slab, representation=slab_rep)
    ifcopenshell.api.spatial.assign_container(model, relating_structure=ifcspace, products=[slab])
    
    return slab


# define a function to add a room to the model, which is just an ifcspace with physical geometry assigned
def add_room(length, width, name, origin):
    
    room_dict = {}

    [ifcspace, space] = add_space(length=length, width=width, name=name, origin=origin+np.array([length/2, width/2, 0]))

    room_dict[name+"wallN"] = add_wall(length=length, name=name+"wallN", origin=origin+np.array([0, width, 0]), rotate=False, ifcspace=ifcspace)
    room_dict[name+"wallS"] = add_wall(length=length, name=name+"wallS", origin=origin+np.array([0,-0.2,0]), rotate=False, ifcspace=ifcspace)

    room_dict[name+"wallE"] = add_wall(length=width+0.4, name=name+"wallE", origin=origin+np.array([length+0.2, -0.2, 0]), rotate=True, ifcspace=ifcspace)
    room_dict[name+"wallW"] = add_wall(length=width+0.4, name=name+"wallW", origin=origin+np.array([0, -0.2, 0]), rotate=True, ifcspace=ifcspace)

    room_dict[name+"slab"] = add_slab(length=length+0.4, width=width+0.4, name=name+"slab", origin=origin+np.array([length/2, width/2, -0.2]), ifcspace=ifcspace)   
    
    # add_space(length=length, width=width, name=name, origin=origin+np.array([length/2, width/2, 0]))

    return [space, room_dict]

Function for create openings

In [ ]:
# adds openings, specifically for doors
def add_opening(width, height, name, origin, ifcspace):
    opening = ifcopenshell.api.root.create_entity(model, ifc_class="IfcOpeningElement", name=name)
    opening_rep = ifcopenshell.api.geometry.add_wall_representation(model,
        context=body, length=width, height=height, thickness=0.2)
    ifcopenshell.api.geometry.assign_representation(model, product=opening, representation=opening_rep)

    matrix = np.eye(4)
    matrix[:,3][0:3] = origin

    ifcopenshell.api.geometry.edit_object_placement(model, product=opening, matrix=matrix, is_si=True)

    # issue. I suspect that this will need to select both walls that the opening intersects with
    # i think it's fine, because technically i just need to know which space the opening connects to.
    # plus i am manually placing the openings. it's more for geometry and visuals
    ifcopenshell.api.feature.add_feature(model, feature=opening, element=opening_rep)


    ifcopenshell.api.spatial.assign_container(model, relating_structure=ifcspace, products=[opening])



Functions for extracting information from model

Model Generation Code

In [58]:
a = add_room(length=5, width=4, name="Room1", origin=(0,0,0))
# when adding rooms next to each other, make sure to account for wall thickness so that they are flush with each other.
# there is also way to way to account for shared walls between rooms, but for simplicity we will just add separate walls
# for each room and make sure they are flush with each other
b = add_room(length=3, width=4, name="Room2", origin=(5.2,0,0))
c = add_room(length=8.2, width=2, name="Corridor", origin=(0,4.2,0))
print(a)
print(b)
print(c)


### TODO add openings in wall, then add space boundaries for openings connecting rooms

model.write("Function_Model.ifc")

[#26=IfcSpace('27S0gSRET5QPKQFoxtf4NG',$,'Room1',$,$,#31,$,$,$,$,$), {'Room1wallN': #41=IfcWall('3RNaRrySf1sPpi7In66kMb',$,'Room1wallN',$,$,#62,#57,$,$), 'Room1wallS': #63=IfcWall('14nJFG5Dv8NAKS_C95452c',$,'Room1wallS',$,$,#84,#79,$,$), 'Room1wallE': #85=IfcWall('0_QsxTc4nBJQEoYy1z2zRX',$,'Room1wallE',$,$,#106,#101,$,$), 'Room1wallW': #107=IfcWall('2p$jDaA4fAGumRDhkYeqzR',$,'Room1wallW',$,$,#128,#123,$,$), 'Room1slab': #129=IfcSlab('0dhAVAsW10N8evKG7or9oL',$,'Room1slab',$,$,#148,#143,$,$)}]
[#151=IfcSpace('0ItEgCAET4HvDtBBTx84YN',$,'Room2',$,$,#156,$,$,$,$,$), {'Room2wallN': #166=IfcWall('190uVw479Fng_5AuKOKFF8',$,'Room2wallN',$,$,#187,#182,$,$), 'Room2wallS': #188=IfcWall('1zVryq1Jj0gx3Pneso$dz2',$,'Room2wallS',$,$,#209,#204,$,$), 'Room2wallE': #210=IfcWall('0LZNe55Mj9795JHilQOj8m',$,'Room2wallE',$,$,#231,#226,$,$), 'Room2wallW': #232=IfcWall('1qU8Pq4wnDgfs06G6aLPpS',$,'Room2wallW',$,$,#253,#248,$,$), 'Room2slab': #254=IfcSlab('3Pp6spjlH1G9ly4K_8TV0L',$,'Room2slab',$,$,#273,#268,$,$)